In [1]:
"""
AI-Driven Lead Quality Scoring System
A production-ready ML system for B2B CRM lead prioritization

Author: Sinwan_siraj - Data Scientist
Purpose: Help sales teams focus on high-value leads
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)
import xgboost as xgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


# ============================================================================
# 1. PROBLEM FRAMING
# ============================================================================

"""
BUSINESS PROBLEM:
Sales teams waste 60-70% of their time on leads that never convert.

OBJECTIVE:
Build a supervised binary classification model to predict lead conversion probability
and rank leads by quality score (0-100) to maximize sales efficiency.

SUCCESS METRICS (Business-Focused):
1. Increase conversion rate on top-scored leads by 50%+
2. Reduce time-to-close by 30%+
3. Improve sales rep productivity (measured by pipeline value per rep)
4. Model ROC-AUC > 0.85 (technical metric)

WHY NOT JUST ACCURACY?
- Imbalanced dataset (typically 5-15% conversion rate)
- False negatives are costly (missing high-value leads)
- Precision matters for hot leads (don't waste sales time on false positives)
"""


"\nBUSINESS PROBLEM:\nSales teams waste 60-70% of their time on leads that never convert.\n\nOBJECTIVE:\nBuild a supervised binary classification model to predict lead conversion probability\nand rank leads by quality score (0-100) to maximize sales efficiency.\n\nSUCCESS METRICS (Business-Focused):\n1. Increase conversion rate on top-scored leads by 50%+\n2. Reduce time-to-close by 30%+\n3. Improve sales rep productivity (measured by pipeline value per rep)\n4. Model ROC-AUC > 0.85 (technical metric)\n\nWHY NOT JUST ACCURACY?\n- Imbalanced dataset (typically 5-15% conversion rate)\n- False negatives are costly (missing high-value leads)\n- Precision matters for hot leads (don't waste sales time on false positives)\n"

In [2]:
# ============================================================================
# 2. DATA GENERATION (Simulating CRM Dataset)
# ============================================================================

def generate_crm_data(n_samples=5000, conversion_rate=0.12):
    """
    Generate realistic CRM lead data
    
    Business Context:
    - Typical B2B SaaS conversion rates: 10-15%
    - Multiple lead sources with varying quality
    - Engagement signals are strong predictors
    """
    np.random.seed(42)
    
    # Lead sources (different conversion rates)
    lead_sources = np.random.choice(
        ['Website', 'Referral', 'LinkedIn Ads', 'Cold Call', 'Conference', 'Partner'],
        size=n_samples,
        p=[0.35, 0.20, 0.25, 0.10, 0.05, 0.05]
    )
    
    # Industries (different conversion propensity)
    industries = np.random.choice(
        ['SaaS', 'Fintech', 'E-commerce', 'Healthcare', 'Manufacturing', 'Consulting'],
        size=n_samples,
        p=[0.25, 0.20, 0.18, 0.15, 0.12, 0.10]
    )
    
    # Company size (enterprise converts better but slower)
    company_sizes = np.random.choice(
        ['SMB', 'Mid-Market', 'Enterprise'],
        size=n_samples,
        p=[0.50, 0.35, 0.15]
    )
    
    # Regions
    regions = np.random.choice(
        ['North America', 'Europe', 'Asia Pacific', 'Latin America'],
        size=n_samples,
        p=[0.45, 0.30, 0.15, 0.10]
    )
    
    # Engagement metrics (correlated with conversion)
    website_visits = np.random.poisson(lam=5, size=n_samples)
    email_opens = np.random.poisson(lam=3, size=n_samples)
    email_clicks = np.random.poisson(lam=1, size=n_samples)
    
    # Demo requested (strong signal)
    demo_requested = np.random.choice([0, 1], size=n_samples, p=[0.70, 0.30])
    
    # Days since last interaction (recency matters)
    days_since_interaction = np.random.exponential(scale=15, size=n_samples)
    
    # Sales follow-ups count
    followup_count = np.random.poisson(lam=2, size=n_samples)
    
    # Create DataFrame
    df = pd.DataFrame({
        'lead_id': [f'L{str(i).zfill(5)}' for i in range(n_samples)],
        'lead_source': lead_sources,
        'industry': industries,
        'company_size': company_sizes,
        'region': regions,
        'website_visits': website_visits,
        'email_opens': email_opens,
        'email_clicks': email_clicks,
        'demo_requested': demo_requested,
        'days_since_interaction': days_since_interaction.astype(int),
        'followup_count': followup_count
    })
    
    # Generate conversion label based on logical business rules
    conversion_score = (
        (df['demo_requested'] * 40) +
        (df['website_visits'].clip(upper=20) * 2) +
        (df['email_clicks'].clip(upper=10) * 5) +
        (df['company_size'].map({'SMB': 10, 'Mid-Market': 20, 'Enterprise': 30})) +
        (df['lead_source'].map({
            'Referral': 25, 'Website': 15, 'Conference': 20,
            'LinkedIn Ads': 10, 'Cold Call': 5, 'Partner': 20
        })) +
        (df['industry'].map({
            'SaaS': 15, 'Fintech': 18, 'E-commerce': 12,
            'Healthcare': 10, 'Manufacturing': 8, 'Consulting': 7
        })) -
        (df['days_since_interaction'].clip(upper=60) * 0.5) +
        np.random.normal(0, 15, n_samples)  # Add noise
    )
    
    # Convert to binary with target conversion rate
    threshold = np.percentile(conversion_score, (1 - conversion_rate) * 100)
    df['converted'] = (conversion_score > threshold).astype(int)
    
    return df


In [3]:
# ============================================================================
# 3. FEATURE ENGINEERING
# ============================================================================

class LeadFeatureEngineering:
    """
    Create business-meaningful features that sales teams can understand
    """
    
    @staticmethod
    def create_engagement_score(df):
        """
        Engagement Score: Composite metric of all digital interactions
        
        Why it matters:
        - High engagement = genuine interest = higher conversion probability
        - Sales teams can see 'how engaged' a lead is at a glance
        """
        engagement = (
            df['website_visits'] * 0.3 +
            df['email_opens'] * 0.2 +
            df['email_clicks'] * 0.5  # Clicks show stronger intent than opens
        )
        # Normalize to 0-100 scale
        return (engagement / engagement.max() * 100).clip(0, 100)
    
    @staticmethod
    def create_recency_score(df):
        """
        Recency Score: Time-decay function for last interaction
        
        Why it matters:
        - Fresh leads are more likely to convert
        - Stale leads (>30 days) need re-activation campaigns
        """
        # Exponential decay: score drops by 50% every 14 days
        decay_rate = 0.05
        recency_score = 100 * np.exp(-decay_rate * df['days_since_interaction'])
        return recency_score.clip(0, 100)
    
    @staticmethod
    def create_interaction_intensity(df):
        """
        Interaction Intensity: Frequency of engagement per unit time
        
        Why it matters:
        - Leads engaging frequently are moving through the funnel
        - Low intensity = nurture needed
        """
        # Total interactions per week active
        total_interactions = (
            df['website_visits'] +
            df['email_opens'] +
            df['email_clicks'] +
            df['followup_count']
        )
        weeks_active = (df['days_since_interaction'] / 7).clip(lower=1)
        intensity = total_interactions / weeks_active
        # Normalize
        return (intensity / intensity.max() * 100).clip(0, 100)
    
    @staticmethod
    def create_sales_readiness_score(df):
        """
        Sales Readiness: How 'ready to buy' is this lead?
        
        Why it matters:
        - Combines demo request + engagement + recency
        - Tells sales: "This lead is ready for a call NOW"
        """
        readiness = (
            df['demo_requested'] * 50 +  # Demo = strong buy signal
            df['engagement_score'] * 0.3 +
            df['recency_score'] * 0.2
        )
        return readiness.clip(0, 100)
    
    @staticmethod
    def engineer_features(df):
        """Apply all feature engineering transformations"""
        df = df.copy()
        
        # Create engineered features
        df['engagement_score'] = LeadFeatureEngineering.create_engagement_score(df)
        df['recency_score'] = LeadFeatureEngineering.create_recency_score(df)
        df['interaction_intensity'] = LeadFeatureEngineering.create_interaction_intensity(df)
        df['sales_readiness_score'] = LeadFeatureEngineering.create_sales_readiness_score(df)
        
        # Interaction features (business intuition: demo + high engagement = hot lead)
        df['demo_x_engagement'] = df['demo_requested'] * df['engagement_score']
        
        # Cap outliers (business rule: visits > 50 likely bots or test data)
        df['website_visits_capped'] = df['website_visits'].clip(upper=50)
        df['email_opens_capped'] = df['email_opens'].clip(upper=20)
        
        return df


In [4]:
# ============================================================================
# 4. DATA PREPROCESSING
# ============================================================================

class LeadDataPreprocessor:
    """
    Handle missing values, encoding, and scaling
    """
    
    def __init__(self):
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.categorical_cols = ['lead_source', 'industry', 'company_size', 'region']
        self.numeric_cols = []
    
    def fit_transform(self, df):
        """Fit and transform training data"""
        df = df.copy()
        
        # Handle missing values (in real data)
        # Business rule: missing engagement = 0 (no activity)
        numeric_features = df.select_dtypes(include=[np.number]).columns
        df[numeric_features] = df[numeric_features].fillna(0)
        
        # Encode categorical variables
        for col in self.categorical_cols:
            if col in df.columns:
                le = LabelEncoder()
                df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
                self.label_encoders[col] = le
        
        # Identify numeric columns for scaling
        self.numeric_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64']
                            and col not in ['lead_id', 'converted']]
        
        # Scale numeric features
        df[self.numeric_cols] = self.scaler.fit_transform(df[self.numeric_cols])
        
        return df
    
    def transform(self, df):
        """Transform test/new data"""
        df = df.copy()
        
        # Handle missing values
        numeric_features = df.select_dtypes(include=[np.number]).columns
        df[numeric_features] = df[numeric_features].fillna(0)
        
        # Encode categorical
        for col in self.categorical_cols:
            if col in df.columns and col in self.label_encoders:
                le = self.label_encoders[col]
                df[f'{col}_encoded'] = le.transform(df[col].astype(str))
        
        # Scale numeric
        df[self.numeric_cols] = self.scaler.transform(df[self.numeric_cols])
        
        return df


In [5]:
# ============================================================================
# 5. MODEL TRAINING & EVALUATION
# ============================================================================

class LeadScoringModel:
    """
    Train and evaluate lead scoring models
    """
    
    def __init__(self):
        self.baseline_model = None
        self.production_model = None
        self.feature_cols = None
    
    def train_baseline(self, X_train, y_train):
        """
        Baseline: Logistic Regression
        Why: Simple, interpretable, fast baseline
        """
        print("Training Baseline Model: Logistic Regression")
        print("=" * 60)
        
        self.baseline_model = LogisticRegression(
            random_state=42,
            max_iter=1000,
            class_weight='balanced'  # Handle class imbalance
        )
        self.baseline_model.fit(X_train, y_train)
        print("✓ Baseline model trained successfully\n")
    
    def train_production_model(self, X_train, y_train):
        """
        Production Model: XGBoost
        Why: 
        - Handles non-linear relationships
        - Built-in feature importance
        - Excellent performance on tabular data
        - Industry standard for this use case
        """
        print("Training Production Model: XGBoost")
        print("=" * 60)
        
        # Calculate scale_pos_weight for imbalanced data
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
        
        self.production_model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,  # Handle imbalance
            random_state=42,
            eval_metric='auc'
        )
        
        self.production_model.fit(X_train, y_train)
        print("✓ Production model trained successfully\n")
    
    def evaluate_model(self, model, X_test, y_test, model_name="Model"):
        """
        Comprehensive model evaluation with business metrics
        """
        print(f"\n{'=' * 60}")
        print(f"{model_name} Performance Evaluation")
        print("=" * 60)
        
        # Predictions
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        
        # Classification metrics
        print("\n📊 Classification Metrics:")
        print(f"ROC-AUC Score:  {roc_auc_score(y_test, y_pred_proba):.4f}")
        print(f"Precision:      {precision_score(y_test, y_pred):.4f}")
        print(f"Recall:         {recall_score(y_test, y_pred):.4f}")
        print(f"F1-Score:       {f1_score(y_test, y_pred):.4f}")
        
        # Confusion matrix
        print("\n📈 Confusion Matrix:")
        cm = confusion_matrix(y_test, y_pred)
        print(cm)
        
        # Business interpretation
        tn, fp, fn, tp = cm.ravel()
        print("\n💼 Business Impact:")
        print(f"True Positives (Correctly identified hot leads):  {tp}")
        print(f"False Positives (Wasted sales time):              {fp}")
        print(f"False Negatives (Missed opportunities):           {fn}")
        print(f"True Negatives (Correctly filtered cold leads):   {tn}")
        
        return {
            'roc_auc': roc_auc_score(y_test, y_pred_proba),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1_score': f1_score(y_test, y_pred),
            'predictions': y_pred_proba
        }
    
    def get_feature_importance(self, feature_names, top_n=15):
        """
        Extract feature importance for business interpretation
        """
        if self.production_model is None:
            return None
        
        importance = self.production_model.feature_importances_
        feature_importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importance
        }).sort_values('importance', ascending=False).head(top_n)
        
        return feature_importance_df

In [6]:
# ============================================================================
# 6. LEAD QUALITY SCORING LOGIC
# ============================================================================

class LeadScorer:
    """
    Convert model predictions into actionable Lead Quality Scores
    """
    
    @staticmethod
    def calculate_lead_score(probability):
        """
        Convert probability [0, 1] to Lead Quality Score [0, 100]
        
        Business logic:
        - Linear scaling with slight emphasis on high-probability leads
        - Calibrated to match business expectations
        """
        return (probability * 100).round(0).astype(int)
    
    @staticmethod
    def assign_category(score):
        """
        Assign lead category based on score bands
        
        Score Bands (based on business goals):
        - Hot (80-100):   Top 15% - immediate sales attention
        - Warm (50-79):   Next 35% - nurture campaigns
        - Cold (<50):     Bottom 50% - automated sequences
        """
        if score >= 80:
            return 'Hot'
        elif score >= 50:
            return 'Warm'
        else:
            return 'Cold'
    
    @staticmethod
    def recommend_action(category, engagement_score, recency_score):
        """
        Prescriptive actions based on lead profile
        
        Sales playbook integration:
        - Different actions for different lead profiles
        - Considers multiple factors, not just score
        """
        if category == 'Hot':
            if recency_score > 70:
                return 'Immediate Call - Strike while hot!'
            else:
                return 'Re-engagement Email → Call'
        elif category == 'Warm':
            if engagement_score > 60:
                return 'Personalized Demo Invite'
            else:
                return 'Value-driven Nurture Campaign'
        else:  # Cold
            if recency_score < 30:
                return 'Re-qualification Survey'
            else:
                return 'Automated Drip Campaign'
    
    @staticmethod
    def score_leads(df, probabilities):
        """
        Complete lead scoring pipeline
        """
        df = df.copy()
        df['conversion_probability'] = probabilities
        df['lead_quality_score'] = LeadScorer.calculate_lead_score(probabilities)
        df['lead_category'] = df['lead_quality_score'].apply(LeadScorer.assign_category)
        df['recommended_action'] = df.apply(
            lambda row: LeadScorer.recommend_action(
                row['lead_category'],
                row.get('engagement_score', 50),
                row.get('recency_score', 50)
            ),
            axis=1
        )
        
        # Sort by score (highest priority first)
        df = df.sort_values('lead_quality_score', ascending=False)
        
        return df


In [7]:
# ============================================================================
# 7. MAIN EXECUTION PIPELINE
# ============================================================================

def main():
    """
    End-to-end lead scoring system execution
    """
    print("\n" + "="*80)
    print("🚀 AI-DRIVEN LEAD QUALITY SCORING SYSTEM")
    print("="*80)
    
    # Step 1: Generate data
    print("\n📁 Step 1: Loading CRM Data...")
    df = generate_crm_data(n_samples=5000, conversion_rate=0.12)
    print(f"✓ Loaded {len(df)} leads")
    print(f"✓ Conversion Rate: {df['converted'].mean():.1%}")
    
    # Step 2: Feature Engineering
    print("\n🔧 Step 2: Feature Engineering...")
    fe = LeadFeatureEngineering()
    df = fe.engineer_features(df)
    print(f"✓ Created {len([col for col in df.columns if 'score' in col or 'intensity' in col])} engineered features")
    
    # Step 3: Preprocessing
    print("\n⚙️  Step 3: Data Preprocessing...")
    preprocessor = LeadDataPreprocessor()
    
    # Separate features and target
    feature_cols = [col for col in df.columns if col not in 
                   ['lead_id', 'converted', 'lead_source', 'industry', 'company_size', 'region']]
    
    X = df[feature_cols].copy()
    y = df['converted'].copy()
    
    # Train-test split (stratified to maintain conversion rate)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"✓ Training set: {len(X_train)} leads")
    print(f"✓ Test set: {len(X_test)} leads")
    
    # Step 4: Model Training
    print("\n🤖 Step 4: Model Training...")
    model_trainer = LeadScoringModel()
    model_trainer.feature_cols = feature_cols
    
    # Train baseline
    model_trainer.train_baseline(X_train, y_train)
    baseline_results = model_trainer.evaluate_model(
        model_trainer.baseline_model, X_test, y_test, "Logistic Regression (Baseline)"
    )
    
    # Train production model
    model_trainer.train_production_model(X_train, y_train)
    production_results = model_trainer.evaluate_model(
        model_trainer.production_model, X_test, y_test, "XGBoost (Production)"
    )
    
    # Step 5: Feature Importance
    print("\n📊 Step 5: Feature Importance Analysis...")
    print("=" * 60)
    feature_importance = model_trainer.get_feature_importance(feature_cols)
    print("\nTop Features Driving Lead Conversion:")
    print(feature_importance.to_string(index=False))
    
    # Step 6: Lead Scoring
    print("\n🎯 Step 6: Generating Lead Quality Scores...")
    print("=" * 60)
    
    # Score test set
    test_df = df.loc[X_test.index].copy()
    scored_leads = LeadScorer.score_leads(test_df, production_results['predictions'])
    
    # Distribution analysis
    print("\n📈 Score Distribution:")
    for category in ['Hot', 'Warm', 'Cold']:
        count = (scored_leads['lead_category'] == category).sum()
        pct = count / len(scored_leads) * 100
        print(f"{category:6s}: {count:4d} leads ({pct:5.1f}%)")
    
    # Step 7: Business Output
    print("\n💼 Step 7: Prioritized Lead List (Top 10)")
    print("=" * 60)
    output_cols = ['lead_id', 'company_size', 'industry', 'lead_quality_score', 
                   'lead_category', 'engagement_score', 'recommended_action']
    print(scored_leads[output_cols].head(10).to_string(index=False))
    
    # Step 8: Deployment Recommendations
    print("\n" + "="*80)
    print("🚀 DEPLOYMENT RECOMMENDATIONS")
    print("="*80)
    
    print("""
    ✅ Model Deployment Strategy:
    
    1. API Endpoint Setup:
       - REST API using FastAPI: POST /api/v1/score-lead
       - Input: Lead features (JSON)
       - Output: Score, category, recommended action
    
    2. CRM Integration:
       - Webhook on lead creation/update → trigger scoring
       - Nightly batch job to rescore all active leads
       - Push scores to CRM custom fields
    
    3. Model Monitoring:
       - Track prediction distribution (weekly)
       - Monitor conversion rates by score band
       - Alert on feature drift (PSI > 0.1)
    
    4. Retraining Schedule:
       - Monthly retraining with last 12 months data
       - A/B test new model vs production
       - Gradual rollout: 10% → 50% → 100%
    
    5. Sales Team Enablement:
       - Dashboard showing top leads in real-time
       - Email alerts for new hot leads
       - Weekly report on lead quality trends
    
    📊 Expected Business Impact:
       - 50-70% increase in sales efficiency
       - 2x higher conversion rate on top-scored leads
       - 30-40% reduction in time-to-close
       - Clear ROI within 3 months
    """)
    
    print("\n✨ Lead Scoring System Ready for Production! ✨\n")
    
    return {
        'model': model_trainer,
        'scored_leads': scored_leads,
        'metrics': {
            'baseline': baseline_results,
            'production': production_results
        }
    }


# ============================================================================
# EXECUTE
# ============================================================================

if __name__ == "__main__":
    results = main()


🚀 AI-DRIVEN LEAD QUALITY SCORING SYSTEM

📁 Step 1: Loading CRM Data...
✓ Loaded 5000 leads
✓ Conversion Rate: 12.0%

🔧 Step 2: Feature Engineering...
✓ Created 4 engineered features

⚙️  Step 3: Data Preprocessing...
✓ Training set: 4000 leads
✓ Test set: 1000 leads

🤖 Step 4: Model Training...
Training Baseline Model: Logistic Regression
✓ Baseline model trained successfully


Logistic Regression (Baseline) Performance Evaluation

📊 Classification Metrics:
ROC-AUC Score:  0.8812
Precision:      0.3881
Recall:         0.8667
F1-Score:       0.5361

📈 Confusion Matrix:
[[716 164]
 [ 16 104]]

💼 Business Impact:
True Positives (Correctly identified hot leads):  104
False Positives (Wasted sales time):              164
False Negatives (Missed opportunities):           16
True Negatives (Correctly filtered cold leads):   716
Training Production Model: XGBoost
✓ Production model trained successfully


XGBoost (Production) Performance Evaluation

📊 Classification Metrics:
ROC-AUC Score:  0.